In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# ---------------------------------------------------------
# Bước 1: Đọc dữ liệu đã qua tiền xử lý
# ---------------------------------------------------------
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

# Xác định cột cần dự báo (Target)
target_col = 'Price' 

# Tách Features (X) và Target (y)
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test: X={X_test.shape}, y={y_test.shape}")

# ---------------------------------------------------------
# Bước 2: Huấn luyện mô hình Baseline (Linear Regression)
# ---------------------------------------------------------
print("\nMô hình Linear Regression")
model = LinearRegression()
model.fit(X_train, y_train)

# ---------------------------------------------------------
# Bước 3: Dự báo trên tập Test
# ---------------------------------------------------------
y_pred_log = model.predict(X_test)

# KHẮC PHỤC LỖI OVERFLOW (TRÀN SỐ)
# Cắt bỏ các giá trị dự báo quá vô lý (giới hạn log_price từ 0 đến 25)
# (25 tương đương e^25 = 72 tỷ tỷ, quá đủ cho giá nhà lớn nhất)
y_pred_log_clipped = np.clip(y_pred_log, a_min=0, a_max=25)

# ---------------------------------------------------------
# Bước 4: INVERSE TRANSFORM (Biến đổi ngược từ Log về Tiền Thật)
# ---------------------------------------------------------
y_train_real = np.expm1(y_train) 
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log_clipped) # Sử dụng biến đã cắt ngọn

# ---------------------------------------------------------
# Bước 5: Tính toán các Metrics dựa trên GIÁ TRỊ TIỀN THẬT
# ---------------------------------------------------------
mae = mean_absolute_error(y_test_real, y_pred_real)
mse = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse) 
r2 = r2_score(y_test_real, y_pred_real)
mape = mean_absolute_percentage_error(y_test_real, y_pred_real)

print("\nKẾT QUẢ ĐÁNH GIÁ BASELINE")
print(f"MAE  (Mean Absolute Error): {mae:,.2f}")
print(f"RMSE (Root Mean Squared Error): {rmse:,.2f}")
print(f"R^2  (Hệ số xác định): {r2:.4f}")
print(f"MAPE (Sai số phần trăm trung bình): {mape * 100:.2f}%")

# ---------------------------------------------------------
# Bước 6: Phân tích MAE theo phân khúc giá tiền thật
# ---------------------------------------------------------
print("\nMAE THEO PHÂN KHÚC GIÁ THỰC TẾ")
results_df = pd.DataFrame({
    'Actual_Price': y_test_real,
    'Predicted_Price': y_pred_real
})

def categorize_price(price):
    # Chia theo mốc tỷ đồng (Ví dụ: 5 tỷ, 15 tỷ)
    if price < 5.0:
        return '1. Phân khúc Thấp (< 5)'
    elif price <= 15.0:
        return '2. Phân khúc Trung bình (5 - 15)'
    else:
        return '3. Phân khúc Cao (> 15)'

results_df['Segment'] = results_df['Actual_Price'].apply(categorize_price)
grouped = results_df.groupby('Segment')

for segment, group in grouped:
    segment_mae = mean_absolute_error(group['Actual_Price'], group['Predicted_Price'])
    print(f"Phân khúc {segment} (Số lượng: {len(group):,} mẫu): MAE = {segment_mae:,.2f}")

Train: X=(40929, 55), y=(40929,)
Test: X=(10233, 55), y=(10233,)

Mô hình Linear Regression

KẾT QUẢ ĐÁNH GIÁ BASELINE
MAE  (Mean Absolute Error): 14,176,688.85
RMSE (Root Mean Squared Error): 1,006,664,354.06
R^2  (Hệ số xác định): -23740.0649
MAPE (Sai số phần trăm trung bình): 750467.58%

MAE THEO PHÂN KHÚC GIÁ THỰC TẾ
Phân khúc 1. Phân khúc Thấp (< 5) (Số lượng: 2 mẫu): MAE = 8,322.94
Phân khúc 2. Phân khúc Trung bình (5 - 15) (Số lượng: 1 mẫu): MAE = 9,157.48
Phân khúc 3. Phân khúc Cao (> 15) (Số lượng: 10,230 mẫu): MAE = 14,180,843.71


In [4]:
train_df.describe()

,Area,Width,Length,Bedrooms,Bathrooms,Floors,Alley Width,Agent Listing Count,is_luxury,Price
count,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,40929.000000
mean,4.982427e-17,4.617859e-17,3.419994e-17,4.930346e-17,-7.556103e-17,3.055426e-17,-1.562434e-17,3.150908e-17,2.508574e-17,9.114637
std,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.486902
min,-7.425334e-03,-5.500898e-02,-5.202595e-02,-7.475987e-02,-1.262410e+00,-4.949458e-03,-3.597625e-01,-4.492501e-01,-9.047551e-01,1.064711
25%,-7.417501e-03,-3.535249e-02,-2.106968e-02,-1.629388e-02,-9.241815e-02,-4.948358e-03,-5.284458e-02,-4.396703e-01,-9.047551e-01,8.101981
50%,-7.412882e-03,-3.024181e-02,-2.106968e-02,-1.629388e-02,-9.241815e-02,-4.948358e-03,-5.284458e-02,-3.875139e-01,-9.047551e-01,8.853808
75%,-7.387577e-03,-1.962730e-02,-2.106968e-02,-1.629388e-02,-9.241815e-02,-4.948358e-03,-5.284458e-02,-7.516695e-03,1.105272e+00,9.903538
max,1.585404e+02,1.965099e+02,1.265873e+02,2.016914e+02,6.308712e+01,2.023066e+02,8.722695e+01,6.279999e+00,1.105272e+00,19.219207


In [2]:
import pandas as pd
train_df = pd.read_csv('../data/processed/train.csv')
print(train_df.columns.tolist())

['Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width', 'Agent Listing Count', 'is_luxury', 'District_Huyện Bình', 'District_Huyện Cần', 'District_Huyện Củ', 'District_Huyện Hóc', 'District_Huyện Nhà', 'District_Huyện Thanh', 'District_Quận 1', 'District_Quận 10', 'District_Quận 11', 'District_Quận 12', 'District_Quận 2', 'District_Quận 3', 'District_Quận 4', 'District_Quận 5', 'District_Quận 6', 'District_Quận 7', 'District_Quận 8', 'District_Quận 9', 'District_Quận Bình', 'District_Quận Gò', 'District_Quận Phú', 'District_Quận Thủ', 'District_Quận Tân', 'District_TP. Hồ', 'District_Unknown', 'Property Type_Kho, nhà xưởng', 'Property Type_Khách sạn', 'Property Type_Nhà riêng', 'Property Type_Nhà trọ', 'Property Type_Văn phòng', 'Property Type_Đất', 'Position_Unknown', 'Position_Đường chính', 'Direction_Nam', 'Direction_Tây', 'Direction_Tây Bắc', 'Direction_Tây Nam', 'Direction_Unknown', 'Direction_Đông', 'Direction_Đông Bắc', 'Direction_Đông Nam', 'Road Type_Đườn